# Original Python model comparison
Coursework archive with outputs cleared and local paths adjusted for this repository. The original row split may put different stays from one patient in both sets. For the later patient-separated benchmark, see train.py.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score
import pickle
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


## 1. Partition data into 50% training set and 50% test set.

In [ ]:
# Read the cohort parquet file
cohort = pd.read_parquet(DATA_DIR / "mimiciv_icu_cohort.parquet")

# Split and stratify, using all columns
train, test = train_test_split(
    cohort,
    test_size=0.5,
    random_state=203,
    stratify=cohort['los_long']
)

# Sort rows by patient, admission and stay
sort_cols = ['subject_id', 'hadm_id', 'stay_id']
train = train.sort_values(sort_cols)
test = test.sort_values(sort_cols)

train.to_parquet(DATA_DIR / "mimiciv_icu_train.parquet", index=False)
test.to_parquet(DATA_DIR / "mimiciv_icu_test.parquet", index=False)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

## 2.Data preprocessing and feature engineering.

In [ ]:
# Separate columns by type
numeric_features = [
    'anchor_age', 'wbc', 'potassium', 'sodium', 'creatinine', 'bicarbonate', 'hematocrit',
    'chloride', 'glucose', 'heart_rate', 'temperature_fahrenheit',
    'respiratory_rate', 'non_invasive_blood_pressure_systolic',
    'non_invasive_blood_pressure_diastolic'
]
categorical_features = ['gender', 'marital_status', 'race']

features = np.concatenate([numeric_features, categorical_features])

X_train = train[features]
y_train = train.los_long
# Test X and y
X_test = test[features]
y_test = test.los_long

In [ ]:
# Transformer for categorical variables
categorical_tf = Pipeline(steps = [
  ("cat_impute", SimpleImputer(strategy = 'most_frequent')),
  ("encoder", OneHotEncoder(drop = 'first'))
])

# Transformer for continuous variables
numeric_tf = Pipeline(steps = [
  ("num_impute", SimpleImputer(strategy = 'mean')),
])

# Column transformer
col_tf = ColumnTransformer(transformers = [
  ('num', numeric_tf, numeric_features),
  ('cat', categorical_tf, categorical_features)
])

## 3.Logistic Regression 

1. Model

In [ ]:
logit_mod = LogisticRegression(
    penalty='elasticnet',
    C=1.0,
    l1_ratio=0.0,
    solver='saga',
    random_state=203,
    max_iter=10000      
)

logit_mod

2. pipeline

In [ ]:
pipe_lr = Pipeline(steps = [
  ("col_tf", col_tf),
  ("model", logit_mod)
  ])
pipe_lr

3. Tuning grid

In [ ]:
# Tune hyper-parameter(s)
C_grid = np.logspace(-1.5, 2, num=10)
l1_ratio_grid = np.linspace(0, 1, 5)
tuned_parameters_lr = {
    "model__C": C_grid,
    "model__l1_ratio": l1_ratio_grid
}
tuned_parameters_lr 

4. Cross-validation 

In [ ]:
search_lr = RandomizedSearchCV(
    pipe_lr,
    tuned_parameters_lr,
    n_iter=10,  
    cv=3,      
    scoring="roc_auc",
    n_jobs=8,
    random_state=203
)

In [ ]:
# Fit CV
tic = time.time()
search_lr.fit(X_train, y_train)
toc = time.time()
print('Execution time:', toc - tic, 'seconds')

5. Visualize CV results.

In [ ]:
cv_res = pd.DataFrame({
  "C": np.array(search_lr.cv_results_["param_model__C"]),
  "auc": search_lr.cv_results_["mean_test_score"],
  "l1_ratio": search_lr.cv_results_["param_model__l1_ratio"]
  })

plt.figure()
sns.relplot(
  # kind = "line",
  data = cv_res,
  x = "C",
  y = "auc",
  hue = "l1_ratio"
  ).set(
    xlabel = "C",
    ylabel = "CV AUC",
    xscale = "log"
);
plt.show()

In [ ]:
# Best CV AUC:
search_lr.best_score_

In [ ]:
# The training accuracy
accuracy_score(
  y_train,
  search_lr.best_estimator_.predict(X_train)
  )

6. Finalize our model

In [ ]:
lr_best=search_lr.best_estimator_

In [ ]:
# The final AUC on the test set 
roc_auc_score(
  y_test,
  lr_best.predict_proba(X_test)[:, 1]
  )

In [ ]:
# The final classification accuracy on the test set 
accuracy_score(
  y_test,
  lr_best.predict(X_test)
  )

In [ ]:
# Report precision, recall, F1 and support by class
target_names = ['No Long Stay', 'Long Stay']
y_test_pred_lr = lr_best.predict(X_test)
print(classification_report(y_test, y_test_pred_lr, target_names=target_names))

7. Save model

In [ ]:
with open(MODEL_DIR / "logit_model.pkl", "wb") as f:
    pickle.dump(lr_best, f)

## 4.Random Forest

1. Model

In [ ]:
rf_mod = RandomForestClassifier(
  # Number of trees
  n_estimators = 100, 
  criterion = 'gini',
  # Number of features to use in each split
  max_features = 'sqrt',
  oob_score = True,
  random_state = 425
  )

2. pipeline

In [ ]:
pipe_rf = Pipeline(steps = [
  ("col_tf", col_tf),
  ("model", rf_mod)
  ])
pipe_rf

3. Tuning grid

In [ ]:
# Tune hyper-parameter(s)
tuned_parameters_rf = {
    "model__n_estimators": [1000,1500,2000],
    "model__max_features": ['log2'],
    "model__max_depth": [None, 10, 20, 30], 
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}
tuned_parameters_rf 

4. Cross-validation

In [ ]:
# Set up CV
search_rf = RandomizedSearchCV(
    pipe_rf,
    tuned_parameters_rf,
    n_iter=10,  
    cv=3,      
    scoring="roc_auc",
    n_jobs=8,
    random_state=203
)

In [ ]:
# Fit CV
tic = time.time()
search_rf.fit(X_train, y_train)
toc = time.time()
print('Execution time:', toc - tic, 'seconds')

5. Visualize CV results

In [ ]:
cv_res = pd.DataFrame({
  "B": np.array(search_rf.cv_results_["param_model__n_estimators"]),
  "auc": search_rf.cv_results_["mean_test_score"],
  "m": search_rf.cv_results_["param_model__max_features"]
  })

plt.figure()
sns.relplot(
  # kind = "line",
  data = cv_res,
  x = "B",
  y = "auc",
  hue = "m"
  ).set(
    xlabel = "B",
    ylabel = "CV AUC"
);
plt.show()

In [ ]:
# Best CV AUC
search_rf.best_score_

In [ ]:
# The training accuracy
accuracy_score(
  y_train,
  search_rf.best_estimator_.predict(X_train)
  )

6. Finalize our model

In [ ]:
rf_best = search_rf.best_estimator_

In [ ]:
# The final AUC on the test set
roc_auc_score(
  y_test,
  rf_best.predict_proba(X_test)[:, 1]
  )

In [ ]:
# The final classification accuracy on the test set
accuracy_score(
  y_test,
  rf_best.predict(X_test)
  )

In [ ]:
# Report precision, recall, F1 and support by class
target_names = ['No Long Stay', 'Long Stay']
y_test_pred_rf = rf_best.predict(X_test)
print(classification_report(y_test, y_test_pred_rf, target_names=target_names))

7. Save model

In [ ]:
with open(MODEL_DIR / "rf_model.pkl", "wb") as f:
    pickle.dump(rf_best, f)

## 5.XGBoost

1. Model

In [ ]:
gb_mod = xgb.XGBClassifier(
  n_estimators = 1000,
  max_depth = 3,
  learning_rate = 0.001,
  random_state = 425
)

2. pipeline

In [ ]:
pipe_gb = Pipeline(steps = [
  ("col_tf", col_tf),
  ("model", gb_mod)
  ])
pipe_gb

3. Tuning grid

In [ ]:
# Tune hyper-parameter(s)
max_depth_grid = [1, 2, 3, 4, 5]
learning_rate_grid = np.logspace(-3, 0, num = 10)
tuned_parameters_gb = {
  "model__max_depth": max_depth_grid,
  "model__learning_rate": learning_rate_grid
  }
tuned_parameters_gb 

4. Cross-validation

In [ ]:
# Set up CV
search_gb = RandomizedSearchCV(
    pipe_gb,
    tuned_parameters_gb,
    n_iter=20,  
    cv=5,      
    scoring="roc_auc",
    n_jobs=8,
    random_state=203
)

In [ ]:
# Fit CV
tic = time.time()
search_gb.fit(X_train, y_train)
toc = time.time()
print('Execution time:', toc - tic, 'seconds')

6. Visualize CV results

In [ ]:
cv_res = pd.DataFrame({
  "learning_rate": np.array(search_gb.cv_results_["param_model__learning_rate"]),
  "auc": search_gb.cv_results_["mean_test_score"],
  "max_depth": search_gb.cv_results_["param_model__max_depth"]
  })

plt.figure()
sns.relplot(
  # kind = "line",
  data = cv_res,
  x = "learning_rate",
  y = "auc",
  hue = "max_depth"
  ).set(
    xlabel = "Learning Rate",
    ylabel = "CV AUC",
    xscale = "log"
);
plt.show()

In [ ]:
# Best CV AUC
search_gb.best_score_

In [ ]:
# The training accuracy 
accuracy_score(
  y_train,
  search_gb.best_estimator_.predict(X_train)
  )

6. Finalize our model

In [ ]:
gb_best = search_gb.best_estimator_

In [ ]:
# The final AUC on the test set
roc_auc_score(
  y_test,
  gb_best.predict_proba(X_test)[:, 1]
  )

In [ ]:
# The final classification accuracy on the test set 
accuracy_score(
  y_test,
  gb_best.predict(X_test)
  )

In [ ]:
# Report precision, recall, F1 and support by class
target_names = ['No Long Stay', 'Long Stay']
y_test_pred_gb = gb_best.predict(X_test)
print(classification_report(y_test, y_test_pred_gb, target_names=target_names))

7. Save model

In [ ]:
with open(MODEL_DIR / "xgb_model.pkl", "wb") as f:
    pickle.dump(gb_best, f)